In [ ]:
import pandas as pd
from sklearn.utils import shuffle
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler

In [ ]:
def split_set(dfs):
    big_df = pd.concat(dfs)
    big_df = shuffle(big_df)
    test_len = len(big_df) // 5
    test_set = big_df[:test_len]
    train_set = big_df[test_len+1:]
    return test_set, train_set

In [ ]:
esa_21_22_fast_actions_parsed = pd.read_pickle('esa_21_22_fast_actions_parsed.pkl')
esa_22_23_fast_actions_parsed = pd.read_pickle('esa_22_23_fast_actions_parsed.pkl')
len(esa_21_22_fast_actions_parsed)

In [ ]:
len(esa_22_23_fast_actions_parsed)

In [ ]:
test_set, train_set = split_set([esa_21_22_fast_actions_parsed, esa_22_23_fast_actions_parsed])
print(len(test_set))
len(train_set)

In [ ]:
test_set['was_shot'] = [1 if (shots > 0) else 0 for shots in test_set['num_of_shot'].values]
len(test_set[test_set['was_shot'] == 1])

In [ ]:
train_set['was_shot'] = [1 if (shots > 0) else 0 for shots in train_set['num_of_shot'].values]
len(train_set[train_set['was_shot'] == 1])

In [ ]:
train_set.columns

In [ ]:
# def set_good_end_values(df):
#     opponent_penalty_area = ['K3', 'K4', 'K5', 'K6', 'L3', 'L4', 'L5', 'L6']
#     newDf = df
#     newDf['good_end'] = [1 if (end_area in opponent_penalty_area) or (shots > 0) else 0 for end_area, shots in zip(df['end_area'].values, df['num_of_shot'].values)]
#     return newDf
    

In [ ]:
# euro_matches = set_good_end_values(euro_matches)
# fifa_matches = set_good_end_values(fifa_matches)
# len(euro_matches[euro_matches['good_end'] == 1])

In [ ]:
def get_sets_for_model(df_train, df_test):
    df_train = shuffle(df_train)
    df_test = shuffle(df_test)

    # scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))

    df_train_ids = df_train['id'].values  # type: ignore
    df_test_ids = df_test['id'].values  # type: ignore
    df_train_teams = df_train['team'].values  # type: ignore
    df_test_teams = df_test['team'].values  # type: ignore
    df_train_was_shot = df_train['was_shot'].values  # type: ignore
    df_test_was_shot = df_test['was_shot'].values  # type: ignore
    # df_train_good_ends = df_train['good_end'].values  # type: ignore
    # df_test_good_ends = df_test['good_end'].values  # type: ignore

    df_train = df_train.drop(columns=['id', 'team', 'path', 'start_area', 'end_area', 'was_shot']) # type: ignore
    df_test = df_test.drop(columns=['id', 'team', 'path', 'start_area', 'end_area', 'was_shot']) # type: ignore

    df_train_columns = df_train.columns.sort_values()
    df_test_columns = df_test.columns.sort_values()

    # df_train = scaler.fit_transform(df_train)
    # df_test = scaler.fit_transform(df_test)

    df_train = pd.DataFrame(df_train, columns=df_train_columns)
    df_test = pd.DataFrame(df_test, columns=df_test_columns)

    df_train['id'] = df_train_ids
    df_test['id'] = df_test_ids
    df_train['team'] = df_train_teams
    df_test['team'] = df_test_teams
    df_train['was_shot'] = df_train_was_shot
    df_test['was_shot'] = df_test_was_shot

    y_train = df_train['was_shot']
    X_train = df_train.drop(columns=['was_shot'])

    y_test = df_test['was_shot']
    X_test = df_test.drop(columns=['was_shot'])

    return X_train, X_test, y_train, y_test

In [ ]:
X_train, X_test, y_train, y_test = get_sets_for_model(train_set, test_set)

In [ ]:
from xgboost import XGBClassifier

def train_model(X_train, X_test, y_train):
    model = XGBClassifier(objective='binary:logistic')

    input_X_train = X_train.drop(columns=['id', 'team', 'num_of_shot'])
    model.fit(input_X_train, y_train)

    input_X_test = X_test.drop(columns=['id', 'team', 'num_of_shot'])
    XGB_predictions = model.predict(input_X_test)

    return model, XGB_predictions, input_X_test

model, pred, input_test = train_model(X_train, X_test, y_train)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

def evaluate_model(y_test, pred):
    print('Confusion matrix')
    print(confusion_matrix(y_test, pred))
    print('Classification report')
    print(classification_report(y_test, pred))
    print('Accuracy= ', accuracy_score(y_test, pred))

evaluate_model(y_test, pred)

In [ ]:
from datetime import datetime

# datetime object containing current date and time
now = datetime.now()
 
print("now =", now)

# dd/mm/YY H:M:S
dt_string = now.strftime("%d/%m/%Y_%H:%M:%S")
print("date and time =", dt_string)

In [ ]:
def feature_importances_raport(input_X_test, model, set_title):
    result = pd.DataFrame(columns=['name', 'value'])
    result['name'] = input_X_test.columns
    result['value'] = model.feature_importances_

    result = result.sort_values(by=['value'], ascending=False)

    result.to_csv(f'features_{set_title}.csv')
    print(result)
    return result

feature_importances = feature_importances_raport(input_test, model, f'fast_sequences_experiment_{dt_string}')

In [ ]:
def generate_scores(model, X_test, X_train, y_train, set_title):
    model = XGBClassifier(objective='binary:logistic')

    input_X_train = X_train.drop(columns=['id', 'team', 'num_of_shot'])
    model.fit(input_X_train, y_train)
    
    input_X_test = X_test.drop(columns=['id', 'team', 'num_of_shot'])
    model = model.predict_proba(input_X_test)

    X_test['score_positive'] = [m[1] for m in model]
    X_test['score_negative'] = [m[0] for m in model]

    X_test.to_csv(f'{set_title}_evaluated.csv')

    return X_test

df_with_scores = generate_scores(model, X_test, X_train, y_train, f'fast_sequences_experiment_{dt_string}')

In [ ]:
df_with_scores